In [0]:
%run ../common/schema

In [0]:
# ADLS paths
ADLS_ACCOUNT   = "adbstoragev10"
ADLS_CONTAINER = "finops-cost-usage-report"
BRONZE_CONTAINER = "bronze"

ADLS_ROOT      = f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net"
BRONZE_ROOT    = f"abfss://{BRONZE_CONTAINER}@{ADLS_ACCOUNT}.dfs.core.windows.net"

# Source paths — where Azure drops the exports
FOCUS_SOURCE_PATH = f"{ADLS_ROOT}/cur/focus-cost"
NON_FOCUS_SOURCE_PATH = f"{ADLS_ROOT}/cur/actual-cost"

# Target paths — where Bronze Delta tables will live
FOCUS_BRONZE_PATH     = f"{BRONZE_ROOT}/focus"
NON_FOCUS_BRONZE_PATH = f"{BRONZE_ROOT}/non_focus"

# Checkpoint paths — Auto Loader tracks processed files here
FOCUS_CHECKPOINT      = f"{BRONZE_ROOT}/_checkpoints/focus"
NON_FOCUS_CHECKPOINT  = f"{BRONZE_ROOT}/_checkpoints/non_focus"

# Unity Catalog
CATALOG = "finops"
SCHEMA  = "bronze"

print("✅ Config done")
print(f"   FOCUS source  : {FOCUS_SOURCE_PATH}")
print(f"   Actual source : {NON_FOCUS_CHECKPOINT}")
# print(f"   Bronze target : {ADLS_ROOT}/bronze/")

In [0]:
from pyspark.sql import functions as F

# Ingest focus using autoloader
df_focus = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", FOCUS_CHECKPOINT) \
  .option("cloudFiles.useNotifications", "false") \
  .option("recursiveFileLookup", "true") \
  .option("pathGlobFilter", "*.csv.gz") \
  .option("header", "true") \
  .schema(FOCUS_SCHEMA) \
  .option("multiline", "true") \
  .option("quote", '"') \
  .option("escape", '"') \
  .load(FOCUS_SOURCE_PATH) \
  .withColumn("_source_file", F.expr("_metadata.file_path")) \
  .withColumn("_ingestion_timestamp", F.current_timestamp())
  
  #.option("inferSchema", "true") \ 
  # .option("mode", "PERMISSIVE") \
  # .option("columnNameOfCorruptRecord", "_corrupt_record") \
  # .schema(FOCUS_SCHEMA.add("_corrupt_record", StringType(), True)) \


# df_focus.status
# df_focus.lastProgress


In [0]:
# write to bronze ADLS
df_focus.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", FOCUS_CHECKPOINT) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(FOCUS_BRONZE_PATH)
    # .toTable("finops.bronze.focus")


In [0]:
from pyspark.sql import functions as F
# Ingest non-focus using autoloader
df_non_focus = spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("cloudFiles.schemaLocation", NON_FOCUS_CHECKPOINT) \
  .option("cloudFiles.useNotifications", "false") \
  .option("header", "true") \
  .option("recursiveFileLookup", "true") \
  .option("pathGlobFilter", "*.csv.gz") \
  .schema(NON_FOCUS_SCHEMA) \
  .load(NON_FOCUS_SOURCE_PATH) \
  .withColumn("_source_file", F.expr("_metadata.file_path")) \
  .withColumn("_ingestion_timestamp", F.current_timestamp())

# .schema(NON_FOCUS_SCHEMA) \
df_non_focus.printSchema()
# df_non_focus.count()


In [0]:
# write to bronze ADLS
# query = df_non_focus.writeStream \
#     .format("delta") \
#     .outputMode("append") \
#     .option("checkpointLocation", NON_FOCUS_CHECKPOINT) \
#     .option("mergeSchema", "true") \
#     .trigger(availableNow=True) \
#     .start(NON_FOCUS_BRONZE_PATH)

# query.awaitTermination()
# df_non_focus.count()
df_non_focus.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", NON_FOCUS_CHECKPOINT) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .start(NON_FOCUS_BRONZE_PATH)
    # .toTable("finops.bronze.non_focus")

In [0]:
%sql
-- DROP TABLE IF EXISTS finops.bronze.focus;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS finops.bronze.focus
LOCATION 'abfss://bronze@adbstoragev10.dfs.core.windows.net/focus';

CREATE TABLE IF NOT EXISTS finops.bronze.non_focus
LOCATION 'abfss://bronze@adbstoragev10.dfs.core.windows.net/non_focus';

In [0]:
%sql
select * from finops.bronze.non_focus limit 3
-- select distinct date from finops.bronze.non_focus
-- select distinct x_InvoiceId from finops.bronze.focus


In [0]:
%sql
-- SHOW EXTERNAL LOCATIONS;
select * from finops.bronze.focus limit 3